> **Solution** for `07-band-limited-wavetables.ipynb` — one possible answer. Work the scaffold first.

# 07 · Additive timbre & band-limited wavetables

Companion to module 07. Run top to bottom. Each step has a cell for you to fill in (it calls
`todo()` until you do) followed by a **check cell** that prints ✓ or ✗ with a hint.

The aha this notebook is heading for: the expensive part of additive synthesis can move to
boot time, and band-limiting becomes a question of *which table*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from synthlib import FS, play, plot_spectrum, spectrum_db, check, todo, recording, read_wav

N = 1024                            # table length
x = np.arange(N) * 2 * np.pi / N    # one period of phase, 0 .. 2π

## 1 · Build waveforms from harmonic weights

A spread is a function *n → amplitude*, with *n* = 1 the fundamental. Write the three
classic ones, and `additive()`, which sums `weight(n) · sin(n·x + phase_n)` over
`harmonics` harmonics into one period.

In [ ]:
def saw_weight(n):
    return 1.0 / n

def square_weight(n):
    return 1.0 / n if n % 2 else 0.0

def triangle_weight(n):
    return (-1) ** ((n - 1) // 2) / n**2 if n % 2 else 0.0

def additive(weight, harmonics, phases=None):
    """One period (N samples) of Σ weight(n)·sin(n·x + phases[n-1]), n = 1..harmonics. No normalization."""
    phases = np.zeros(harmonics) if phases is None else phases
    return sum(weight(n) * np.sin(n * x + phases[n - 1]) for n in range(1, harmonics + 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
ideal = {"saw": (np.pi - x) / 2, "square": np.pi / 4 * np.sign(np.sin(x)), "triangle": np.pi**2 / 8 * (2 / np.pi) * np.arcsin(np.sin(x))}
for ax, (name, w) in zip(axes, [("saw", saw_weight), ("square", square_weight), ("triangle", triangle_weight)]):
    ax.plot(x, ideal[name], color="gray", lw=1, label="ideal")
    ax.plot(x, additive(w, 64), lw=1, label="64 harmonics")
    ax.set_title(name); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
saw = additive(saw_weight, 64)
ideal_saw = (np.pi - x) / 2
away = (x > 0.3) & (x < 2 * np.pi - 0.3)
check(np.sqrt(np.mean((saw[away] - ideal_saw[away]) ** 2)) < 0.05,
      "a 64-harmonic saw matches (π − x)/2 away from the jump", "check saw_weight and additive")
overshoot = (saw.max() - np.pi / 2) / np.pi
check(0.07 < overshoot < 0.10, f"Gibbs overshoot is {overshoot:.1%} of the jump (theory ≈ 9%)")
check(np.allclose(additive(square_weight, 3), np.sin(x) + np.sin(3 * x) / 3), "the square uses odd harmonics only")
check(np.allclose(additive(triangle_weight, 3), np.sin(x) - np.sin(3 * x) / 9), "the triangle alternates signs on 1/n²")

## 2 · Alias on purpose

A *naïve* sawtooth is just a phase ramp — it contains every harmonic, including the ones
above Nyquist (22 050 Hz), which fold back. Write the ramp and `alias_of()`, which says where
a component at `f` lands after sampling.

In [ ]:
def naive_saw(f, seconds=1.0, fs=FS):
    """A raw phase ramp rising from −1 to 1, f times per second. No band-limiting."""
    t = np.arange(int(seconds * fs)) / fs
    return 2 * ((f * t) % 1.0) - 1

def alias_of(f, fs=FS):
    """The frequency (0 .. fs/2) that a component at f Hz appears at after sampling at fs."""
    f = f % fs
    return fs - f if f > fs / 2 else f

In [ ]:
f0 = 3520
ax = plot_spectrum(naive_saw(f0), f0=f0, label=f"naive saw {f0} Hz")
for k in range(7, 13):   # the harmonics above Nyquist, and where they fold to
    ax.annotate(f"{k}×", (alias_of(k * f0), -15), color="C3", fontsize=8, ha="center")
ax.set_title("dotted = true harmonics · red = folded ones"); plt.show()

In [ ]:
check(alias_of(30000) == 14100 and alias_of(24640) == 19460 and alias_of(1000) == 1000,
      "alias_of folds around fs/2", "f mod fs, then mirror anything above fs/2")
f, m = spectrum_db(naive_saw(3520))
inharmonic = np.abs(((f + 1760) % 3520) - 1760) > 50   # more than 50 Hz from any multiple of 3520
check(m[inharmonic].max() > -40, f"the naive saw has inharmonic junk up to {m[inharmonic].max():.0f} dB")

## 3 · Fix it

Keep only harmonics with `n·f < fs/2`. Use amplitude `2/(π n)` and a minus sign so it lines
up with the naïve ramp (a rising ramp is −(2/π) Σ sin(2π n f t)/n). Then listen to both — the
naïve one has a faint metallic whistle the clean one doesn't.

In [ ]:
def bandlimited_saw(f, seconds=1.0, fs=FS):
    """The same rising saw, with only the harmonics below fs/2."""
    t = np.arange(int(seconds * fs)) / fs
    n_max = int((fs / 2) // f)
    return -(2 / np.pi) * sum(np.sin(2 * np.pi * n * f * t) / n for n in range(1, n_max + 1))

In [ ]:
for name, sig in [("naive", naive_saw(3520, 2)), ("band-limited", bandlimited_saw(3520, 2))]:
    print(name); play(0.3 * sig, normalize=False)
f, m = spectrum_db(bandlimited_saw(3520))
inharmonic = np.abs(((f + 1760) % 3520) - 1760) > 50
check(m[inharmonic].max() < -60, f"band-limited saw: worst inharmonic component is {m[inharmonic].max():.0f} dB")
t = np.arange(400) / FS
check(np.corrcoef(naive_saw(3520)[:400], bandlimited_saw(3520)[:400])[0, 1] > 0.9,
      "it's the same waveform as the naïve ramp, just band-limited", "check the sign and 2/(π n)")

## 4 · Mip-map design

One table per octave: level *L* covers base frequencies `[80·2^L, 80·2^(L+1))`. A table may
keep only the harmonics that stay under Nyquist across its whole octave — if it's built for
its **top** edge. Built for its **bottom** edge it's brighter, but the top of the octave
aliases.

In [ ]:
BASE_HZ = 80.0

def harmonics_for_level(level, edge="top", fs=FS):
    """How many harmonics level `level` may keep when it must be alias-free at its
    'top' or 'bottom' edge."""
    f = BASE_HZ * 2 ** (level + (1 if edge == "top" else 0))
    return max(1, int((fs / 2) // f))

In [ ]:
print("level  octave (Hz)        top-edge  bottom-edge")
for L in range(9):
    lo, hi = BASE_HZ * 2**L, BASE_HZ * 2**(L + 1)
    print(f"{L:>5}  {lo:>6.0f} – {hi:<7.0f}  {harmonics_for_level(L, 'top'):>8}  {harmonics_for_level(L, 'bottom'):>11}")
check(harmonics_for_level(3, "top") == 17 and harmonics_for_level(3, "bottom") == 34
      and harmonics_for_level(0, "top") == 137, "harmonic counts per level")

The cost of top-edge limiting: just above a boundary a note loses half its harmonics.
Compare a note just below 640 Hz (level 2, 34 harmonics) with one just above (level 3, 17).

In [ ]:
def saw_with(f, n_harm, seconds=1.0):
    t = np.arange(int(seconds * FS)) / FS
    return -(2 / np.pi) * sum(np.sin(2 * np.pi * n * f * t) / n for n in range(1, n_harm + 1))

below, above = saw_with(630, harmonics_for_level(2)), saw_with(650, harmonics_for_level(3))
ax = plot_spectrum(below, label="630 Hz, level 2"); plot_spectrum(above, ax=ax, label="650 Hz, level 3"); plt.show()
print("630 Hz (level 2)"); play(0.3 * saw_with(630, harmonics_for_level(2), 1.5), normalize=False)
print("650 Hz (level 3)"); play(0.3 * saw_with(650, harmonics_for_level(3), 1.5), normalize=False)

## 5 · Normalization: worst case vs measured peak

Dividing by Σ|w(n)| guarantees the table can't exceed full scale (all harmonics in phase).
Real sums peak much lower. Compute *peak ÷ bound* for each spread — the headroom you'd waste
by stopping at the worst case.

In [ ]:
SPREADS = {
    "sine": lambda n: 1.0 if n == 1 else 0.0,
    "saw": saw_weight,
    "square": square_weight,
    "triangle": triangle_weight,
    "octaves": lambda n: 1 / n if n & (n - 1) == 0 else 0.0,
}

def peak_over_bound(weight, harmonics=64, phases=None):
    """Peak |value| of the built table divided by Σ|weight(n)|."""
    table = additive(weight, harmonics, phases)
    bound = sum(abs(weight(n)) for n in range(1, harmonics + 1))
    return np.abs(table).max() / bound

In [ ]:
rng = np.random.default_rng(1)
random_phases = rng.uniform(0, 2 * np.pi, 64)
print("spread     aligned  random phases")
for name, w in SPREADS.items():
    print(f"{name:<9}  {peak_over_bound(w):6.2f}  {peak_over_bound(w, phases=random_phases):6.2f}")
check(abs(peak_over_bound(SPREADS["sine"]) - 1) < 1e-3, "a sine uses its whole bound")
check(peak_over_bound(saw_weight) < 0.6, f"a saw peaks at {peak_over_bound(saw_weight):.2f} of its bound — rescale to the measured peak")

Random phases change the peak — but do they change the sound? Listen (220 Hz, 64 harmonics):

In [ ]:
def render(weight, f, harmonics=64, phases=None, seconds=1.5):
    t = np.arange(int(seconds * FS)) / FS
    phases = np.zeros(harmonics) if phases is None else phases
    y = sum(weight(n) * np.sin(2 * np.pi * n * f * t + phases[n - 1]) for n in range(1, harmonics + 1) if n * f < FS / 2)
    return 0.3 * y / np.abs(y).max()

print("aligned phases"); play(render(saw_weight, 220), normalize=False)
print("random phases");  play(render(saw_weight, 220, phases=random_phases), normalize=False)

## 6 · RAM

Each table is `table_len` int16 samples. Write the arithmetic, then fit your design under a
128 KB budget.

In [ ]:
def table_bytes(spreads, levels, table_len=1024, sample_bytes=2):
    return spreads * levels * table_len * sample_bytes

In [ ]:
check(table_bytes(5, 6) == 61440, "5 spreads × 6 levels × 1024 × 2 B = 61 440 B")

def levels_needed(f_max):
    """Levels 0..L where L is the level of the highest frequency any voice reaches."""
    return int(np.floor(np.log2(f_max / BASE_HZ))) + 1

for f_max in (1000, 2000, 4000):
    print(f"top note {f_max:>4} Hz: {levels_needed(f_max)} levels, "
          f"{table_bytes(12, levels_needed(f_max)) / 1024:.0f} KB for 12 spreads "
          f"(vs {table_bytes(12, 9) / 1024:.0f} KB building all 9)")

## 7 · Verify the real thing

Record your device sweeping its brightest spread slowly from the lowest to the highest
pitch and save it as `recordings/07-saw-sweep.wav` (see `recordings/README.md`). Aliases show
up as faint lines moving *against* the pitch. Until you have a recording, this shows a naïve
(aliasing) sweep so you know what to look for.

In [ ]:
rec = recording("07-saw-sweep.wav")
if rec:
    y, fs = read_wav(rec)
    title = f"your recording: {rec.name}"
else:
    fs, seconds = FS, 4.0
    f = 80 * (2000 / 80) ** (np.arange(int(seconds * fs)) / (seconds * fs))   # log sweep 80 Hz → 2 kHz
    phase = np.cumsum(f) / fs
    y = 2 * (phase % 1.0) - 1
    title = "no recording yet — a naïve saw sweep, aliasing on purpose"
plt.figure(figsize=(10, 4))
plt.specgram(y, NFFT=2048, Fs=fs, noverlap=1536, cmap="magma", vmin=-120)
plt.ylim(0, fs / 2); plt.xlabel("s"); plt.ylabel("Hz"); plt.title(title); plt.show()

## What did you find?

Write your conclusion in the module's notes panel, then compare:

> The expensive part of additive synthesis can move to boot time, and band-limiting becomes
> a question of *which table*.